In [24]:
import pandas as pd
import numpy as np 

In [25]:
df = pd.read_csv("cleaned_dataset.csv")
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [26]:
df["promo_code_used"].unique()

array(['Yes', 'No'], dtype=object)

## Dependency Score

This feature estimates how dependent a customer is on promotions and discounts.

### Category Logic
- 0 → No promo code and no discount used
- 1 → Either promo code OR discount used
- 2 → Both promo code and discount used

### Business Purpose
To identify whether customer purchases are driven by genuine brand loyalty or promotional incentives.

In [27]:
df['dependency_score'] = (
    df['promo_code_used'].map({'Yes':1, 'No':0}) +
    df['discount_applied'].map({'Yes':1, 'No':0})
)

## Value Tier

### Category Logic 1
This feature classifies customers based on **purchase amount** and **previous purchases**.

**Value Score =**
- 60% Purchase Amount (normalized)
- 40% Previous Purchases (normalized)

Customers are then divided into:
- Bottom 33% → Low Value
- Middle 33% → Medium Value
- Top 33% → High Value

### Business Purpose
To identify valuable customers based on both their spending and repeat purchase behavior.

---

### Category Logic 2
This feature classifies customers using **purchase amount, previous purchases, purchase frequency, and subscription status**.

**Value Score =**
- 45% Purchase Amount (normalized)
- 30% Previous Purchases (normalized)
- 15% Frequency of Purchases (encoded & normalized)
- 10% Subscription Status

Customers are then divided into:
- Bottom 33% → Low Value
- Middle 33% → Medium Value
- Top 33% → High Value

### Business Purpose
To identify customer value using spending, repeat purchases, shopping frequency, and subscription status, providing a more comprehensive measure of customer value.

---

### Selected Logic
**Category Logic 2** was selected because it considers multiple dimensions of customer value rather than relying only on spending and repeat purchases, resulting in a more comprehensive customer segmentation.


**Category Logic 1 Code**

In [28]:
from sklearn.preprocessing import MinMaxScaler

# Create a copy
df_logic1 = df.copy()

# Normalize purchase amount and previous purchases
scaler = MinMaxScaler()

df_logic1[['purchase_score', 'repeat_score']] = scaler.fit_transform(
    df_logic1[['purchase_amount_usd', 'previous_purchases']]
)

# Calculate Value Score
df_logic1['value_score'] = (
    0.6 * df_logic1['purchase_score'] +
    0.4 * df_logic1['repeat_score']
)

# Create Value Tier
df_logic1['value_tier'] = pd.qcut(
    df_logic1['value_score'],
    q=3,
    labels=['Low Value', 'Medium Value', 'High Value']
)

# View results
df_logic1[['purchase_amount_usd', 'previous_purchases',
           'value_score', 'value_tier']].head()

,purchase_amount_usd,previous_purchases,value_score,value_tier
0,53,14,0.353622,Low Value
1,64,2,0.338163,Low Value
2,73,23,0.577092,Medium Value
3,90,49,0.916837,High Value
4,49,31,0.462398,Medium Value


**Category logic 2 code**

In [29]:
from sklearn.preprocessing import MinMaxScaler

# Create a copy
df_logic2 = df.copy()

# Encode Subscription Status
df_logic2['subscription_score'] = df_logic2['subscription_status'].map({
    'Yes': 1,
    'No': 0
})

# Encode Frequency of Purchases
frequency_map = {
    'Weekly': 7,
    'Bi-Weekly': 6,
    'Fortnightly': 5,
    'Monthly': 4,
    'Every 3 Months': 3,
    'Quarterly': 2,
    'Annually': 1
}

df_logic2['frequency_score'] = df_logic2['frequency_of_purchases'].map(frequency_map)

# Normalize continuous variables
scaler = MinMaxScaler()

cols = ['purchase_amount_usd',
        'previous_purchases',
        'frequency_score']

df_logic2[['purchase_score',
           'repeat_score',
           'frequency_score_norm']] = scaler.fit_transform(df_logic2[cols])

# Calculate Value Score
df_logic2['value_score'] = (
    0.45 * df_logic2['purchase_score'] +
    0.30 * df_logic2['repeat_score'] +
    0.15 * df_logic2['frequency_score_norm'] +
    0.10 * df_logic2['subscription_score']
)

# Create Value Tier
df_logic2['value_tier'] = pd.qcut(
    df_logic2['value_score'],
    q=3,
    labels=['Low Value', 'Medium Value', 'High Value']
)

# View results
df_logic2[['purchase_amount_usd',
           'previous_purchases',
           'frequency_of_purchases',
           'subscription_status',
           'value_score',
           'value_tier']].head()

,purchase_amount_usd,previous_purchases,frequency_of_purchases,subscription_status,value_score,value_tier
0,53,14,Fortnightly,Yes,0.465217,Medium Value
1,64,2,Fortnightly,Yes,0.453622,Medium Value
2,73,23,Weekly,Yes,0.682819,High Value
3,90,49,Weekly,Yes,0.937628,High Value
4,49,31,Annually,Yes,0.446798,Medium Value


In [30]:
df['value_tier'] = df_logic2['value_tier']
df['value_score'] = df_logic2['value_score']
df[['value_score', 'value_tier']].head()

,value_score,value_tier
0,0.465217,Medium Value
1,0.453622,Medium Value
2,0.682819,High Value
3,0.937628,High Value
4,0.446798,Medium Value


## Satisfaction Flag

This feature indicates whether the customer appears satisfied based on review ratings.

### Category Logic
- Review Rating ≥ 4 → Satisfied
- Review Rating < 4 → Unsatisfied

### Business Purpose
To identify dissatisfied customers and potential retention issues.

In [31]:
df['satisfaction_flag'] = df['review_rating'].apply(
    lambda x: 'Satisfied' if x >= 4 else 'Unsatisfied'
)

In [32]:
df['satisfaction_flag'].value_counts()

satisfaction_flag
Unsatisfied    2266
Satisfied      1634
Name: count, dtype: int64

## Loyalty Segment

### Category Logic
This feature measures customer loyalty using:
- Previous Purchases (positive contribution)
- Dependency Score (negative contribution)

Customers with many previous purchases and low promotional dependency are considered more loyal.

Customers are divided into:
- Low Loyalty
- Medium Loyalty
- High Loyalty

### Business Purpose
To identify customers who repeatedly purchase due to genuine brand preference rather than promotional incentives.

In [33]:
# Create a copy (optional)
df_loyalty = df.copy()

# Normalize previous purchases
df_loyalty['repeat_score'] = (
    (df_loyalty['previous_purchases'] - df_loyalty['previous_purchases'].min()) /
    (df_loyalty['previous_purchases'].max() - df_loyalty['previous_purchases'].min())
)

# Normalize dependency score (0, 1, 2 → 0, 0.5, 1)
df_loyalty['dependency_score_norm'] = (
    df_loyalty['dependency_score'] / 2
)

# Calculate loyalty score
df_loyalty['loyalty_score'] = (
    0.7 * df_loyalty['repeat_score'] -
    0.3 * df_loyalty['dependency_score_norm']
)

# Create Loyalty Segment
df_loyalty['loyalty_segment'] = pd.qcut(
    df_loyalty['loyalty_score'],
    q=3,
    labels=['Low Loyalty', 'Medium Loyalty', 'High Loyalty']
)

# View results
df_loyalty[['previous_purchases',
            'dependency_score',
            'loyalty_score',
            'loyalty_segment']].head()

,previous_purchases,dependency_score,loyalty_score,loyalty_segment
0,14,2,-0.114286,Low Loyalty
1,2,2,-0.285714,Low Loyalty
2,23,2,0.014286,Low Loyalty
3,49,2,0.385714,High Loyalty
4,31,2,0.128571,Medium Loyalty


In [34]:
df['loyalty_segment'] = df_loyalty['loyalty_segment']

## Retention Risk

This feature identifies customers who may be at risk of discontinuing purchases.

### Category Logic
Customer is marked as High Risk when:
- Review Rating < 3
- Dependency Score ≥ 1
- Previous Purchases < 10

Otherwise:
- Low Risk

### Business Purpose
To help the business proactively target customers likely to churn.

In [35]:
df['retention_risk'] = np.where(
    (df['review_rating'] < 3) &
    (df['dependency_score'] >= 1) &
    (df['previous_purchases'] < 10),
    'High Risk',
    'Low Risk'
)

## Age Group

This feature groups customers into age segments.

### Category Logic
- Age ≤ 25 → Young Adult
- Age between 26 and 40 → Adult
- Age > 40 → Mature

### Business Purpose
To identify which age groups contribute most to customer value and retention.

In [36]:
def age_group(age):
    if age <= 25:
        return 'Young Adult'
    elif age <= 40:
        return 'Adult'
    else:
        return 'Mature'

df['age_group'] = df['age'].apply(age_group)

## Promo Sensitivity

This feature classifies customers according to their promotional dependency level.

### Category Logic
- Dependency Score = 0 → Low
- Dependency Score = 1 → Medium
- Dependency Score = 2 → High

### Business Purpose
To identify which customer groups are highly discount-sensitive.

In [37]:
def promo_sensitivity(score):
    if score == 0:
        return 'Low'
    elif score == 1:
        return 'Medium'
    else:
        return 'High'

df['promo_sensitivity'] = df['dependency_score'].apply(promo_sensitivity)

## Premium Customer

This feature flags high-value loyal customers.

### Category Logic
Customer is marked as Premium Customer when:
- Value Tier = High Value
- Loyalty Segment = High Loyalty 

Otherwise:
- No

### Business Purpose
To identify customers who contribute significantly to long-term revenue.

In [38]:
df['premium_customer'] = np.where(
    (df['value_tier'] == 'High Value') &
    (df['loyalty_segment'] == 'High Loyalty'),
    'Yes',
    'No'
)

In [39]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,...,frequency_of_purchases,dependency_score,value_tier,value_score,satisfaction_flag,loyalty_segment,retention_risk,age_group,promo_sensitivity,premium_customer
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Fortnightly,2,Medium Value,0.465217,Unsatisfied,Low Loyalty,Low Risk,Mature,High,No
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Fortnightly,2,Medium Value,0.453622,Unsatisfied,Low Loyalty,Low Risk,Young Adult,High,No
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Weekly,2,High Value,0.682819,Unsatisfied,Low Loyalty,Low Risk,Mature,High,No
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Weekly,2,High Value,0.937628,Unsatisfied,High Loyalty,Low Risk,Young Adult,High,Yes
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Annually,2,Medium Value,0.446798,Unsatisfied,Medium Loyalty,Low Risk,Mature,High,No


In [40]:
df.to_csv("engineered_dataset.csv", index=False)